# R45: one complete supplementary photograph

Choose an **L4 or A100 GPU**, then **Run all**. Return `baomai-additional-front-pilot-r45-results.zip`.

This is a new training-data pilot: reference **137 photos / 3,784 fronts**, candidate **138 photos / 3,827 fronts**. The candidate adds only the complete R44 `frame_0383` with its 43 visible fronts. Both arms keep the exact effective R37 labels for all original photos and independently start from epoch173. Other R44 photos remain held and are not imported.

The hypothesis concerns thin-front localization and cassette/rack boundaries. One extra photograph is a small experiment; it has both corners locatable on every front and is not a dedicated partial-corner treatment. The twenty original development photos stay outside training. The added photo is never used as a test image.

The fixed native recipe uses seed 46, batch 4, input size 1280, FP32, AdamW and 30 epochs. Expected budgets are 1,050 batches and 78 optimizer updates per arm, with 4,110 versus 4,140 image presentations. The runner verifies actual counters and preserves a failed or incompatible run for inspection. Final `last.pt` selection is fixed. No model is automatically promoted.


## 1. Get the pinned source

Only text source, labels and review decisions are checked out. The identical R44 review capsule is rebuilt locally from the already-public PR131 archive. Unrelated historical model/result archives are excluded.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
SOURCE_REVISION = '60bc0b6a3c0efadd53a21dab0699b39327054396'
PROTOCOL_PATH = 'experiments/additional-front-pilot-2026-11-26-r45.json'
PROTOCOL_SHA256 = '7a4acb537abb611647e82615afe96badba2ec60a5a77218a9b3f28ab30cb80ba'
ORIGINAL_ASSET_REVISION = 'a4ead2ee8de56a5e9e125947b78d09b1fce84b92'
SPARSE_PATHS = ['/scripts/', '/metadata/', '/data/manifest.json', '/data/labels/', '/data/candidates/front-review-2026-11-26-r37/', '/data/reviews/additional-fronts-2026-11-26-r44/', '/experiments/additional-front-pilot-2026-11-26-r45.json', '/data/reviews/fronts-2026-11-26-r38/', '/data/references/sequence-refined-reference-2026-11-18/', '/data/references/visible-front-completion-2026-11-14/']
BASE = Path(os.environ.get("BAOMAI_R45_BASE", "/content/baomai-r45" if Path("/content").is_dir() else str(Path.cwd()/"baomai-r45"))).resolve()
REPO, INPUTS, WORK = BASE/"source", BASE/"inputs", BASE/"run"
BASE.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None, env=None):
    subprocess.run([str(x) for x in command], cwd=cwd, env=env, check=True)

git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
if not REPO.exists():
    run(["git", "init", REPO], env=git_env)
    run(["git", "remote", "add", "origin", os.environ.get("BAOMAI_R45_SOURCE_URL", "https://github.com/Siuuuers/dl.git")], cwd=REPO)
    run(["git", "config", "remote.origin.promisor", "true"], cwd=REPO)
    run(["git", "config", "remote.origin.partialclonefilter", "blob:none"], cwd=REPO)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_REVISION], cwd=REPO, env=git_env)
    run(["git", "sparse-checkout", "init", "--no-cone"], cwd=REPO)
    run(["git", "sparse-checkout", "set", "--no-cone", *SPARSE_PATHS], cwd=REPO)
    run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO, env=git_env)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == SOURCE_REVISION, "Existing source uses different pins; preserve it and choose a new BAOMAI_R45_BASE."
assert not subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, text=True).strip(), "Changed tracked source was preserved."
assert hashlib.sha256((REPO/PROTOCOL_PATH).read_bytes()).hexdigest() == PROTOCOL_SHA256
SCRIPT = REPO/"scripts/additional_front_pilot_r45.py"
command = [sys.executable, "-B", SCRIPT, "build-review", "--root", REPO, "--work", BASE/"review-cache"]
if os.environ.get("BAOMAI_R45_R43_ARCHIVE"):
    command += ["--return-archive", os.environ["BAOMAI_R45_R43_ARCHIVE"]]
run(command)
print("Pinned text source ready. R44 capsule rebuilt locally from the public PR131 return.")


## 2. Restore verified originals and stage the pair

Original images and the epoch173 initializer are downloaded into a separate inputs folder and hash-checked. This leaves the repository dataset unchanged.


In [ ]:
run([sys.executable, "-m", "pip", "install", "--quiet", "ultralytics==8.4.182", "psutil"])
import torch
assert torch.cuda.is_available(), "Select an L4 or A100 GPU runtime before the training cells."
print("GPU:", torch.cuda.get_device_name(0))
run([sys.executable, "-B", SCRIPT, "download-inputs", "--root", REPO, "--work", INPUTS, "--asset-revision", ORIGINAL_ASSET_REVISION])
run([sys.executable, "-B", SCRIPT, "prepare", "--root", REPO, "--work", WORK,
     "--images-root", INPUTS, "--initial-weights", INPUTS/"weights/cassette_lower_epoch173.pt"])


## 3. Run a discarded resource trial, then both fresh arms

A completed compatible run is reused. Each full arm starts from the original initializer; the trial never initializes a training arm. Optional Drive backup preserves each completed arm before the next one starts.


In [ ]:
BACKUP = None
if os.environ.get("BAOMAI_R45_USE_DRIVE", "1") == "1":
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        drive.mount("/content/drive")
        BACKUP = Path("/content/drive/MyDrive/baomai-r45")/SOURCE_REVISION
        BACKUP.mkdir(parents=True, exist_ok=True)
        for name in ("resource-trial", "reference", "candidate"):
            if not (WORK/name).exists() and (BACKUP/name).is_dir():
                shutil.copytree(BACKUP/name, WORK/name)
                if (BACKUP/(name+".log")).is_file():
                    shutil.copyfile(BACKUP/(name+".log"), WORK/(name+".log"))
command = [sys.executable, "-B", SCRIPT, "run", "--root", REPO, "--work", WORK, "--device", "0"]
if BACKUP:
    command += ["--backup-dir", BACKUP]
run(command)


## 4. Evaluate and return the pair

Both checkpoints use the same fixed inference settings on the same twenty originals. Current R38, frozen666 and historical663 references are reported separately. The existing optional postclip filter is a secondary view applied equally to both arms.


In [ ]:
run([sys.executable, "-B", REPO/"scripts/evaluate_additional_front_pilot_r45.py", "--root", REPO, "--work", WORK, "--device", "0"])
ARCHIVE = BASE/"baomai-additional-front-pilot-r45-results.zip"
run([sys.executable, "-B", SCRIPT, "package", "--root", REPO, "--work", WORK, "--output", ARCHIVE])
summary = json.loads((WORK/"evaluation/summary.json").read_text())
for view, reports in summary["references"]["r38"].items():
    for role in ("reference", "candidate"):
        d = reports[role]["detection"]
        print(view, role, "misses", d["fn"], "unmatched", d["fp"], "duplicates", d["duplicate_fp"], "other", d["other_fp"])
if BACKUP:
    destination = BACKUP/ARCHIVE.name
    if destination.exists():
        assert hashlib.sha256(destination.read_bytes()).hexdigest() == hashlib.sha256(ARCHIVE.read_bytes()).hexdigest(), "Different saved return was preserved."
    else:
        shutil.copyfile(ARCHIVE, destination)
try:
    from google.colab import files
except ImportError:
    print("Return:", ARCHIVE)
else:
    files.download(str(ARCHIVE))


## Reading the result

Compare misses, duplicate outputs, other unmatched outputs, and corner successes together. A poorly aligned output can create both one miss and one unmatched output. The report retains changed reference identities and zero/one/two-locatable-corner groups; missing eligible targets remain in corner-success denominators.

This pilot has no independent capture-session test. It changes one training photograph, not the target definition, architecture, loss or inference settings. The accepted R38 reference and the historical references use different boundary conventions and must not be pooled. PR35 remains selected under its historical gate; R45 never changes that selection automatically.

The ZIP has its own `baomai-additional-front-pilot-results` schema with both final checkpoints, exact labels, protocol, source hashes, receipts, logs and prediction caches. Use the R45 evaluator for this dataset expansion; the old 137-image checkpoint validator is deliberately not reused.
